In [1]:
from llama_index.core import KnowledgeGraphIndex, SimpleDirectoryReader
from llama_index.core import StorageContext
from llama_index.graph_stores.neo4j import Neo4jGraphStore


from llama_index.llms.openai import OpenAI
from IPython.display import Markdown, display

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
# from fastapi import FastAPI
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

[nltk_data] Downloading package punkt_tab to c:\Users\M LAKSHMI
[nltk_data]     NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-
[nltk_data]     packages\llama_index\core\_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\RAMAYAN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_11276\3885403216.py:26: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


In [2]:
import os
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_DATABASE = "ayodhyakanda"

graph_store = Neo4jGraphStore(
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    url=NEO4J_URI,
    database=NEO4J_DATABASE,
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

In [3]:
# Clean dataset first
ew = graph_store.query(
    """
MATCH (n) detach delete n
"""
)

In [4]:
file = 'ayodhyakanda.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['Prompt_Answer'],
    metadata={
        "canto": i['Canto'],
        "canto_summary": i['Summary'],
        "r_id": str(i['Canto']) + "_" + str(i['Prompt_Number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{key}=>{value}",
    text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents))
print(f"Document Metadata: {documents[-1].metadata}")
storage_context = StorageContext.from_defaults(graph_store=graph_store)

423
Document Metadata: {'canto': '119', 'canto_summary': 'Canto CXIX\n\nGlorifying the evening shades, that had now fallen, Anasuya sends away Sita,\ndecked with the ornaments bestowed on her by herself, to the presence of\nSri Rama. To the latter, who was agreeably surprised to behold Sita\nresplendent with heavenly jewels, she relates how she received the\njewels as a loving gift from the hermitess. Accepting the hospitality\nof the hermits and staying with them for the night, Sri Rama\nasks leave of them in the morning to proceed further\nand, following the route shown by the hermits,\nthe princely party enters the Dandaka forest\n\n', 'r_id': '119_2'}


In [5]:
from llama_index.core.prompts.base import PromptTemplate
from llama_index.core.prompts.prompt_type import PromptType
DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW = (
    "Some text from ancient story Ramayan is provided below. Given the text, extract knowledge graph triplets with the following rules:\n"

    "1. Extract triplets with the following patterns:\n"
    "   (charecter, relation, charecter) - Resolve the relations between charecters based on the context given\n"
    "   (charecter, context, event) - Resolve the relations between charecters and the major events in the context given\n"
    "   (charecter, 'is in', Canto) - Mapping for each charecter to canto\n"
    "2. Focus more on the important events and relations between charecters\n"
    "3. Avoid stopwords. The output should only be triplets and no extra commenting/template literals.\n"
    "4. use snake casing for everything in triplets\n"
    "5. for (charecter, relation, charecter) type of triplets, use the following relations:\n"
    "- is (mother/father/brother/sister/wife/husband/son/daughter/siblings/aunt/uncle/niece/nephew/teacher/mentor/related_in_canto_'number') of\n"
    "6. While generating triplets corresponding to charecter context event, focus only on important event of the given text, no need to split every sentence of text\n"
    "7. The number of triplets triplets of type (charecter, 'is in', Canto) should not exceed 10, and the number of other two type of triplets should not exceed {max_knowledge_triplets} \n"
    "These triplets are used for RAG, so only give important triplets and make sure there is no important data missing\n"
    "Text: {text}\n"
    "Triplets:\n"
)
DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW = PromptTemplate(
    DEFAULT_KG_TRIPLET_EXTRACT_TMPL_NEW,
    prompt_type=PromptType.KNOWLEDGE_TRIPLET_EXTRACT,
)

In [6]:
index1 = KnowledgeGraphIndex.from_documents(
    documents[:100],
    kg_triplet_extract_template = DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW,
    storage_context=storage_context,
    max_triplets_per_chunk=20,
    include_embeddings=True
)

Prompt num:  1
Prompt num:  2
Prompt num:  3
Prompt num:  4
Prompt num:  5
Prompt num:  6
Prompt num:  7
Prompt num:  8
Prompt num:  9
Prompt num:  10
Prompt num:  11
Prompt num:  12
Prompt num:  13
Prompt num:  14
Prompt num:  15
Prompt num:  16
Prompt num:  17
Prompt num:  18
Prompt num:  19
Prompt num:  20
Prompt num:  21
Prompt num:  22
Prompt num:  23
Prompt num:  24
Prompt num:  25
Prompt num:  26
Prompt num:  27
Prompt num:  28
Prompt num:  29
Prompt num:  30
Prompt num:  31
Prompt num:  32
Prompt num:  33
Prompt num:  34
Prompt num:  35
Prompt num:  36
Prompt num:  37
Prompt num:  38
Prompt num:  39
Prompt num:  40
Prompt num:  41
Prompt num:  42
Prompt num:  43
Prompt num:  44
Prompt num:  45
Prompt num:  46
Prompt num:  47
Prompt num:  48
Prompt num:  49
Prompt num:  50
Prompt num:  51
Prompt num:  52
Prompt num:  53
Prompt num:  54
Prompt num:  55
Prompt num:  56
Prompt num:  57
Prompt num:  58
Prompt num:  59
Prompt num:  60
Prompt num:  61
Prompt num:  62
Prompt num:  63
P

In [7]:
index1.storage_context.persist(persist_dir="ayodhyakanda_index1")

In [8]:
index2 = KnowledgeGraphIndex.from_documents(
    documents[100:200],
    kg_triplet_extract_template = DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW,
    storage_context=storage_context,
    max_triplets_per_chunk=20,
    include_embeddings=True
)

Prompt num:  101
Prompt num:  102
Prompt num:  103
Prompt num:  104
Prompt num:  105
Prompt num:  106
Prompt num:  107
Prompt num:  108
Prompt num:  109
Prompt num:  110
Prompt num:  111
Prompt num:  112
Prompt num:  113
Prompt num:  114
Prompt num:  115
Prompt num:  116
Prompt num:  117
Prompt num:  118
Prompt num:  119
Prompt num:  120
Prompt num:  121
Prompt num:  122
Prompt num:  123
Prompt num:  124
Prompt num:  125
Prompt num:  126
Prompt num:  127
Prompt num:  128
Prompt num:  129
Prompt num:  130
Prompt num:  131
Prompt num:  132
Prompt num:  133
Prompt num:  134
Prompt num:  135
Prompt num:  136
Prompt num:  137
Prompt num:  138
Prompt num:  139
Prompt num:  140
Prompt num:  141
Prompt num:  142
Prompt num:  143
Prompt num:  144
Prompt num:  145
Prompt num:  146
Prompt num:  147
Prompt num:  148
Prompt num:  149
Prompt num:  150
Prompt num:  151
Prompt num:  152
Prompt num:  153
Prompt num:  154
Prompt num:  155
Prompt num:  156
Prompt num:  157
Prompt num:  158
Prompt num:  1

In [9]:
index2.storage_context.persist(persist_dir="ayodhyakanda_index2")

In [10]:
index3 = KnowledgeGraphIndex.from_documents(
    documents[200:300],
    kg_triplet_extract_template = DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW,
    storage_context=storage_context,
    max_triplets_per_chunk=20,
    include_embeddings=True
)

Prompt num:  201
Prompt num:  202
Prompt num:  203
Prompt num:  204
Prompt num:  205
Prompt num:  206
Prompt num:  207
Prompt num:  208
Prompt num:  209
Prompt num:  210
Prompt num:  211
Prompt num:  212
Prompt num:  213
Prompt num:  214
Prompt num:  215
Prompt num:  216
Prompt num:  217
Prompt num:  218
Prompt num:  219
Prompt num:  220
Prompt num:  221
Prompt num:  222
Prompt num:  223
Prompt num:  224
Prompt num:  225
Prompt num:  226
Prompt num:  227
Prompt num:  228
Prompt num:  229
Prompt num:  230
Prompt num:  231
Prompt num:  232
Prompt num:  233
Prompt num:  234
Prompt num:  235
Prompt num:  236
Prompt num:  237
Prompt num:  238
Prompt num:  239
Prompt num:  240
Prompt num:  241
Prompt num:  242
Prompt num:  243
Prompt num:  244
Prompt num:  245
Prompt num:  246
Prompt num:  247
Prompt num:  248
Prompt num:  249
Prompt num:  250
Prompt num:  251
Prompt num:  252
Prompt num:  253
Prompt num:  254
Prompt num:  255
Prompt num:  256
Prompt num:  257
Prompt num:  258
Prompt num:  2

In [11]:
index3.storage_context.persist(persist_dir="ayodhyakanda_index3")

In [12]:
index4 = KnowledgeGraphIndex.from_documents(
    documents[300:],
    kg_triplet_extract_template = DEFAULT_KG_TRIPLET_EXTRACT_PROMPT_NEW,
    storage_context=storage_context,
    max_triplets_per_chunk=20,
    include_embeddings=True
)

Prompt num:  301
Prompt num:  302
Prompt num:  303
Prompt num:  304
Prompt num:  305
Prompt num:  306
Prompt num:  307
Prompt num:  308
Prompt num:  309
Prompt num:  310
Prompt num:  311
Prompt num:  312
Prompt num:  313
Prompt num:  314
Prompt num:  315
Prompt num:  316
Prompt num:  317
Prompt num:  318
Prompt num:  319
Prompt num:  320
Prompt num:  321
Prompt num:  322
Prompt num:  323
Prompt num:  324
Prompt num:  325
Prompt num:  326
Prompt num:  327
Prompt num:  328
Prompt num:  329
Prompt num:  330
Prompt num:  331
Prompt num:  332
Prompt num:  333
Prompt num:  334
Prompt num:  335
Prompt num:  336
Prompt num:  337
Prompt num:  338
Prompt num:  339
Prompt num:  340
Prompt num:  341
Prompt num:  342
Prompt num:  343
Prompt num:  344
Prompt num:  345
Prompt num:  346
Prompt num:  347
Prompt num:  348
Prompt num:  349
Prompt num:  350
Prompt num:  351
Prompt num:  352
Prompt num:  353
Prompt num:  354
Prompt num:  355
Prompt num:  356
Prompt num:  357
Prompt num:  358
Prompt num:  3

In [17]:
index4.storage_context.persist(persist_dir="ayodhyakanda_index4")

In [19]:
# from llama_index.core import StorageContext, load_index_from_storage, load_indices_from_storage
# storage_context2 = StorageContext.from_defaults(persist_dir="ayodhyakanda_index4")
# indexi = load_index_from_storage(storage_context2)

In [ ]:
query_engine = index1.as_query_engine(
    include_text=False, response_mode="tree_summarize"
)

response = query_engine.query("How is viswamitra related to rama?")

Prompt num:  427


KeyboardInterrupt: 

: 

In [ ]:
query_engine = index2.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))
response = query_engine.query("who is rama")
print(response)

C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_11276\256977517.py:7: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


Prompt num:  426
Extracted keywords: ['Rama']


KeyboardInterrupt: 

In [ ]:
response = query_engine.query("who are wife of rishyasringa")
print(response)

In [ ]:
response = query_engine.query("who are parents of santa")
print(response)

In [ ]:
response = query_engine.query("who pro|phosised that sage rishyasringa will give birth to dasaratha's son")
print(response)

In [ ]:
response = query_engine.query("name of viswamitra's cow")
print(response)

In [ ]:
response = query_engine.query("which canto rama is avatar of vishnu")
print(response)

In [ ]:
response = query_engine.query("when is it revealed that rama is avatar of vishnu")
print(response)

In [ ]:
query_engine = indexi.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=10,
    verbose=True)
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))
response = query_engine.query("when is it revealed that rama is avatar of vishnu")
print(response)

In [ ]:
response = query_engine.query("when did vishnu promise to reincarnate as rama")
print(response)

In [ ]:
response = query_engine.query("which sage prophosised that dasaratha will have sons")
print(response)

In [ ]:
response = query_engine.query("how many sons does king sagara have")
print(response)

In [ ]:
response = query_engine.query("how many sons does viswamitra lose in battle")
print(response)

In [ ]:
response = query_engine.query("who told valmiki ramayana for the first time")
print(response)